In [ ]:
import os
import gc
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

**EDA**

In [ ]:
print(train.columns)
train.head()
train.shape

In [ ]:
train.head()

In [ ]:
display(train.dtypes)

In [ ]:
train.info()

In [ ]:
print(test.shape)
print(test.columns.tolist())

In [ ]:
display(test.head())

In [ ]:
test.info()

In [ ]:
train.isnull().sum()

In [ ]:
print(train.duplicated().sum())

print(train["prompt"].duplicated().sum())

In [ ]:
duplicate_prompts = train[train["prompt"].duplicated(keep=False)]

display(
    duplicate_prompts[
        ["prompt", "A", "B", "C", "D", "E", "answer"]
    ].sort_values("prompt").head(10)
)

In [ ]:
train["answer"].value_counts()

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(
    data=train,
    x="answer",
    order=["A","B","C","D","E"]
)

plt.title("Distribution of Target Classes")
plt.xlabel("Correct Answer")
plt.ylabel("Count")

plt.show()

print(train["answer"].value_counts())

print("\nPercentage")
print((train["answer"].value_counts(normalize=True)*100).round(2))

In [ ]:
train["prompt_length"] = train["prompt"].str.split().str.len()

train["prompt_length"].describe()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(
    train["prompt_length"],
    bins=30,
    kde=True
)

plt.title("Prompt Length Distribution")

plt.xlabel("Number of Words")

plt.ylabel("Frequency")

plt.show()

In [ ]:
for option in ["A","B","C","D","E"]:
    train[f"{option}_length"] = train[option].str.split().str.len()

In [ ]:
train[
    [
        "A_length",
        "B_length",
        "C_length",
        "D_length",
        "E_length"
    ]
].describe()

In [ ]:
avg_lengths = []

for option in ["A","B","C","D","E"]:
    avg_lengths.append(train[f"{option}_length"].mean())

plt.bar(
    ["A","B","C","D","E"],
    avg_lengths
)

plt.ylabel("Average Words")

plt.title("Average Length of Options")

plt.show()

In [ ]:
words = []

for sentence in train["prompt"]:

    words.extend(sentence.lower().split())

print("Vocabulary Size:", len(set(words)))

In [ ]:
counter = Counter(words)

counter.most_common(20)

In [ ]:
common = counter.most_common(20)

word = [x[0] for x in common]
count = [x[1] for x in common]

plt.figure(figsize=(10,5))

plt.bar(word,count)

plt.xticks(rotation=90)

plt.title("Most Frequent Words")

plt.show()

In [ ]:
train.loc[
    train["prompt_length"].idxmax(),
    "prompt"
]

In [ ]:
train.loc[
    train["prompt_length"].idxmin(),
    "prompt"
]

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 512

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = row["prompt"]

        first_sentences = [prompt] * 5
        second_sentences = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"],
        ]

        enc = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"]
        }

In [ ]:
test_dataset = TestDataset(test, tokenizer, MAX_LENGTH)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

In [ ]:
DATA_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge"

test = pd.read_csv(f"{DATA_PATH}/test.csv")

test.head()

In [ ]:
import os

DEBERTA_PATH = "/kaggle/input/notebooks/arc24ade/deberta-model/models"

for i in range(5):
    print(f"Fold {i}")
    print(os.listdir(f"{DEBERTA_PATH}/fold_{i}"))
    print("-"*40)

In [ ]:
ROBERTA_PATH = "/kaggle/input/notebooks/arc24ade/roberta/models"

for i in range(5):
    print(f"Fold {i}")
    print(os.listdir(f"{ROBERTA_PATH}/fold_{i}"))
    print("-"*40)

In [ ]:
from torch.utils.data import Dataset

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = row["prompt"]

        first_sentences = [prompt] * 5
        second_sentences = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"],
        ]

        encoding = self.tokenizer(
            first_sentences,
            second_sentences,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }

In [ ]:
import numpy as np
import torch

def predict_model(model, dataloader):
    model.eval()

    predictions = []

    with torch.no_grad():
        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            probs = torch.softmax(logits, dim=1)

            predictions.append(probs.cpu().numpy())

    return np.concatenate(predictions)

In [ ]:
from transformers import AutoTokenizer

deberta_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base"
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

In [ ]:
from torch.utils.data import DataLoader

deberta_dataset = MCQDataset(test, deberta_tokenizer)
roberta_dataset = MCQDataset(test, roberta_tokenizer)

deberta_loader = DataLoader(
    deberta_dataset,
    batch_size=8,
    shuffle=False
)

roberta_loader = DataLoader(
    roberta_dataset,
    batch_size=8,
    shuffle=False
)

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(
    "/kaggle/input/notebooks/arc24ade/deberta-model/models/fold_0"
)

model.to(device)

print("Loaded successfully!")

In [ ]:
label_map = np.array(["A", "B", "C", "D", "E"])

In [ ]:
deberta_preds = np.zeros((len(test), 5))

for fold in range(5):

    print(f"DeBERTa Fold {fold}")

    model = AutoModelForMultipleChoice.from_pretrained(
        f"/kaggle/input/notebooks/arc24ade/deberta-model/models/fold_{fold}"
    )

    model.to(device)

    deberta_preds += predict_model(model, deberta_loader)

    del model
    gc.collect()
    torch.cuda.empty_cache()

deberta_preds /= 5

In [ ]:
roberta_preds = np.zeros((len(test), 5))

for fold in range(5):

    print(f"RoBERTa Fold {fold}")

    model = AutoModelForMultipleChoice.from_pretrained(
        f"/kaggle/input/notebooks/arc24ade/roberta/models/fold_{fold}"
    )

    model.to(device)

    roberta_preds += predict_model(model, roberta_loader)

    del model
    gc.collect()
    torch.cuda.empty_cache()

roberta_preds /= 5

In [ ]:
top3 = np.argsort(-roberta_preds, axis=1)[:, :3]

predictions = [
    " ".join(label_map[idx])
    for idx in top3
]

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

In [ ]:
import json
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import re
from torch.utils.data import Dataset, DataLoader

In [ ]:
MODEL_DIR = "/kaggle/input/notebooks/arc24ade/bilstm-model/models"

with open(f"{MODEL_DIR}/vocab.json", "r") as f:
    vocab = json.load(f)

with open(f"{MODEL_DIR}/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

In [ ]:
def tokenize(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()


def numericalize(text):

    ids = []

    for token in tokenize(text):
        ids.append(vocab.get(token, vocab["<UNK>"]))

    ids = ids[:CFG.MAX_LEN]

    if len(ids) < CFG.MAX_LEN:
        ids += [0] * (CFG.MAX_LEN - len(ids))

    return ids

In [ ]:
class TestDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        prompt = str(row["prompt"])

        choices = []

        for option in ["A","B","C","D","E"]:

            text = prompt + " " + str(row[option])

            choices.append(
                numericalize(text)
            )

        return torch.tensor(
            choices,
            dtype=torch.long
        )

In [ ]:
class TestDataset(Dataset):

    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]
        prompt = str(row["prompt"])

        choices = []

        for option in ["A","B","C","D","E"]:

            text = prompt + " " + str(row[option])

            choices.append(
                numericalize(text)
            )

        return torch.tensor(
            choices,
            dtype=torch.long
        )

In [ ]:

test = pd.read_csv(CFG.TEST_PATH)

test_dataset = TestDataset(test)

test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False
)

bilstm_preds = []

model.eval()

with torch.no_grad():

    for choices in test_loader:

        choices = choices.to(CFG.DEVICE)

        logits = model(choices)

        probs = F.softmax(logits, dim=1)

        bilstm_preds.append(probs.cpu().numpy())

bilstm_preds = np.concatenate(bilstm_preds, axis=0)

print(bilstm_preds.shape)
print(bilstm_preds[:5])   